# Farm Agent — 2. Fine-tuning Qwen2.5-0.5B-Instruct on agricultural Q&A

**Settings:** Accelerator = **GPU T4** · Internet = On.

**Secrets** (Add-ons → Secrets — never paste keys into cells):
- `HF_TOKEN` — write token, only needed for the last cell (push merged model to the Hub)
- `GROQ_API_KEY` — optional, enables the LLM-as-judge scores

Set `RUN_COMPARISON = False` below to skip the QLoRA vs LoRA vs full section (~1 h).
Work happens in `/tmp`; only results (+ the small adapter) land in `/kaggle/working`.

In [ ]:
REPO = "https://github.com/shashank-sj/farm-agent.git"
BRANCH = "main"
HUB_REPO = "shank-j/qwen2.5-0.5b-farm"   # where the merged model is pushed (last cell)
RUN_COMPARISON = True
# Code source: a private Kaggle dataset with the repo (attached as input) if present, else GitHub.
import glob, os, shutil, tarfile, zipfile
DST = "/tmp/farm-agent"
shutil.rmtree(DST, ignore_errors=True)
apps = glob.glob("/kaggle/input/**/app.py", recursive=True)
archives = glob.glob("/kaggle/input/**/farm_agent_src.*", recursive=True)
if apps:
    shutil.copytree(os.path.dirname(apps[0]), DST)
elif archives:
    os.makedirs(DST)
    a = archives[0]
    (tarfile.open(a) if a.endswith((".tar.gz", ".tgz")) else zipfile.ZipFile(a)).extractall(DST)
else:
    !git clone --depth 1 -b $BRANCH $REPO $DST
%cd $DST
OUT = "/kaggle/working"
# Kaggle gives two T4s; 4-bit models + the HF Trainer must not be split across GPUs (DataParallel).
%env CUDA_VISIBLE_DEVICES=0

In [ ]:
!pip install -q -r finetune/requirements.txt langchain-text-splitters
# Kaggle preinstalls torchao 0.10; recent PEFT raises on torchao < 0.16 but works without it.
!pip uninstall -y -q torchao

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None
for name in ["HF_TOKEN", "GROQ_API_KEY"]:
    try:
        os.environ[name] = secrets.get_secret(name)
        print(f"{name}: loaded")
    except Exception:
        print(f"{name}: not set (optional)")
JUDGE = "--judge" if os.environ.get("GROQ_API_KEY") else ""

## 1. Data: clean + leakage-safe grouped split

In [ ]:
!python -m finetune.data --out data/finetune

## 2. Baseline: the base model before any fine-tuning

In [ ]:
!python -m finetune.evaluate --label base $JUDGE

## 3. QLoRA fine-tune (main model)

In [ ]:
!python -m finetune.train --method qlora --out outputs/finetune/qlora
!python -m finetune.evaluate --label qlora --adapter outputs/finetune/qlora/final $JUDGE

## 4. Post-training quantisation of the fine-tuned model (memory vs quality)

In [ ]:
!python -m finetune.evaluate --label qlora-int8 --adapter outputs/finetune/qlora/final --load 8bit --n-gen 150
!python -m finetune.evaluate --label qlora-nf4 --adapter outputs/finetune/qlora/final --load 4bit --n-gen 150

## 5. Retrieval-augmented answers (index built from the TRAIN split only, so it can't leak test answers)

In [ ]:
!python -m scripts.build_rag_index --kb-dir none --qa-jsonl data/finetune/train.jsonl --out data/rag_index_train_only
!python -m finetune.evaluate --label base+rag --rag-index data/rag_index_train_only $JUDGE
!python -m finetune.evaluate --label qlora+rag --adapter outputs/finetune/qlora/final --rag-index data/rag_index_train_only $JUDGE

## 6. QLoRA vs LoRA vs full fine-tuning (same data, same settings)

In [ ]:
# Same data, epochs, prompt and mask as the QLoRA run above, so only the method differs.
if RUN_COMPARISON:
    !python -m finetune.train --method lora --out outputs/finetune/lora
    !python -m finetune.evaluate --label lora --adapter outputs/finetune/lora/final $JUDGE
    !python -m finetune.train --method full --out outputs/finetune/full
    !python -m finetune.evaluate --label full --full-model outputs/finetune/full/final $JUDGE

## 7. Summary table + loss curve

In [ ]:
import json
import matplotlib.pyplot as plt

cols = ["answer_perplexity", "rouge_l", "token_f1", "judge_correctness", "judge_helpfulness",
        "avg_answer_words", "sec_per_answer", "gpu_mem_after_load_gb"]
lines = ["| variant | " + " | ".join(cols) + " |", "|---" * (len(cols) + 1) + "|"]
for f in sorted(glob.glob("results/finetune/eval_*.json")):
    r = json.load(open(f))
    lines.append(f"| {r['label']} | " + " | ".join(str(r.get(c, '-')) for c in cols) + " |")
lines += ["", "| run | trainable % | steps | best eval loss | answer ppl (val) | minutes | peak GPU GB |", "|---|---|---|---|---|---|---|"]
for f in sorted(glob.glob("outputs/finetune/*/train_summary.json")):
    s = json.load(open(f))
    lines.append(f"| {f.split('/')[-2]} | {s['trainable_pct']} | {s['global_steps']} | {s['best_eval_loss']} | "
                 f"{s['best_eval_answer_perplexity']} | {s['train_minutes']} | {s['peak_gpu_mem_gb']} |")
summary = "\n".join(lines)
os.makedirs("results/finetune", exist_ok=True)
open("results/finetune/SUMMARY.md", "w").write(summary + "\n")
print(summary)

hist = json.load(open("outputs/finetune/qlora/log_history.json"))
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.plot(*zip(*tr), label="train loss"); plt.plot(*zip(*ev), "o-", label="val loss (answer tokens)")
plt.xlabel("step"); plt.ylabel("cross-entropy"); plt.legend(); plt.title("QLoRA training")
plt.savefig("results/finetune/qlora_loss.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# Small results only (the adapter is ~35 MB); checkpoints and merged weights stay in /tmp
!mkdir -p $OUT/finetune
!cp -r results/finetune/* $OUT/finetune/
!cp data/finetune/data_summary.json $OUT/finetune/
for f in glob.glob("outputs/finetune/*/train_summary.json") + glob.glob("outputs/finetune/*/log_history.json"):
    run = f.split("/")[-2]
    shutil.copy(f, f"{OUT}/finetune/{run}_{os.path.basename(f)}")
shutil.copytree("outputs/finetune/qlora/final", f"{OUT}/finetune/qlora_adapter", dirs_exist_ok=True)

## 8. Merge the adapter and push the standalone model (used as the app's fallback via `LOCAL_LLM_MODEL`)

In [ ]:
if os.environ.get("HF_TOKEN"):
    !python -m finetune.merge_adapter --adapter outputs/finetune/qlora/final --out /tmp/merged --push-to-hub $HUB_REPO
else:
    print("HF_TOKEN not set — skipping push. The adapter is saved in /kaggle/working/finetune/qlora_adapter.")